# Lab 3.13 — Failure Analysis Across Modalities
Compare clean and stressed slices for a transferred image classifier and a transparent order-sensitive sequence task.


In [ ]:
import copy, random, numpy as np, torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
SEED=13; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.set_num_threads(1)
d=load_digits(); X=d.data.astype('float32')/16.0; y=d.target.astype('int64'); Xt,Xv,yt,yv=train_test_split(X,y,test_size=0.25,random_state=SEED,stratify=y)
class E(torch.nn.Module):
    def __init__(self): super().__init__(); self.net=torch.nn.Sequential(torch.nn.Linear(64,32),torch.nn.ReLU())
    def forward(self,x): return self.net(x)
class M(torch.nn.Module):
    def __init__(self,e=None,out=2): super().__init__(); self.encoder=e or E(); self.head=torch.nn.Linear(32,out)
    def forward(self,x): return self.head(self.encoder(x))
def train(m,a,b,n,lr):
    o=torch.optim.SGD([p for p in m.parameters() if p.requires_grad],lr=lr); f=torch.nn.CrossEntropyLoss()
    for _ in range(n): o.zero_grad(); z=f(m(a),b); z.backward(); o.step()
    return m
def acc(m,a,b):
    m.eval();
    with torch.no_grad(): return float((m(a).argmax(1)==b).float().mean())
src=train(M(out=10),torch.tensor(Xt),torch.tensor(yt),120,0.1); e=copy.deepcopy(src.encoder); [setattr(p,'requires_grad',False) for p in e.parameters()]
target=np.isin(yt,[0,6,8,9]).astype('int64'); tv=torch.tensor(np.isin(yv,[0,6,8,9]).astype('int64')); idx=np.arange(len(yt)); small,_=train_test_split(idx,train_size=40,random_state=SEED,stratify=target); model=train(M(e),torch.tensor(Xt[small]),torch.tensor(target[small]),200,0.2); val=torch.tensor(Xv)
clean=acc(model,val,tv); rng=np.random.default_rng(SEED); noisy=torch.tensor(np.clip(Xv+rng.normal(0,0.35,Xv.shape).astype('float32'),0,1)); noisy_acc=acc(model,noisy,tv)
seq=np.array([[1.,2.,4.,7.],[8.,6.,3.,1.],[2.,2.5,3.,5.],[7.,5.,4.,2.]]); labels=(seq[:,-1]>seq[:,0]).astype(int); pred=lambda s:(s[:,-1]>s[:,0]).astype(int); sc=float(np.mean(pred(seq)==labels)); sr=float(np.mean(pred(seq[:,::-1])==labels))
print(f'image_clean_accuracy={clean:.3f}'); print(f'image_noisy_accuracy={noisy_acc:.3f}'); print(f'sequence_clean_accuracy={sc:.3f}'); print(f'sequence_reversed_accuracy={sr:.3f}')
assert sc==1.0 and sr==0.0 and clean>=noisy_acc


## Slice record
Keep perturbation definition, sample count, seed, metric, clean control, and representative failures together.
